In [ ]:
from Code.NetworkExpansion import *
import networkx as nx
import matplotlib.pyplot as plt
from Code.externalize_results import *

# Example 1: Expansions from primordial seed set

write_outputs = False
max_generation = 80

reaction_scopes = {'example': get_sides(kegg_reactions),
                   'element-constrained': restrict_elemental_scope(primordial_seed_set, get_sides(kegg_reactions))}

# run network expansions
results = dict()
for name, reaction_scope in reaction_scopes.items():
    results[name] = expand_from_seed_set(seeds=primordial_seed_set, reactions=reaction_scope, max_gen=max_generation)

# summarize results
def result_to_df(key: str, to_apply):
    return pd.concat({i: results[i][key].apply(to_apply) for i in results.keys()}, axis=1)

summaries = {'compound count': result_to_df(key='cps', to_apply=lambda x: x.sum()),
             'reaction count': result_to_df(key='rns', to_apply=lambda x: x.sum()),
             'chiral fraction': result_to_df(key='chiral_fraction', to_apply=lambda x: x)}
# plot results
fig, ax = plt.subplots(len(summaries.keys()), 1, figsize=(4, 7), sharex=True, layout='compressed')
for ict, iname in enumerate(summaries.keys()):
    summaries[iname].plot(ax=ax[ict], ylabel=iname, cmap='rainbow', grid=True, xlabel='Expansion step').legend(loc='lower right')

# write output files
if write_outputs:
    plt.savefig('primordial_expansion_comparison.png', dpi=300, edgecolor='w', facecolor='w')
    write_output_files(results=results)

plt.show()
plt.close()

In [ ]:
# Example 2: Expansions from all-chiral or all-achiral seed sets.
# For a chiral (or achiral) seed set batch, set query to is_chiral or is_chiral==False, respectively.

max_generation = 80

# chiral or achiral seed set batch
n_samples = 100
seed_sets = {i: list(chiral_labels.query('is_chiral').sample(n=len(primordial_seed_set), random_state=i).index) for i in range(n_samples)}
reaction_scopes = {i: restrict_elemental_scope(j, get_sides(kegg_reactions)) for i,j in seed_sets.items()}

# run network expansions
results = dict()
for name, reaction_scope in reaction_scopes.items():
    results[name] = expand_from_seed_set(seeds=seed_sets[name], reactions=reaction_scope, max_gen=max_generation)

# summarize results
summaries = {'compound count': result_to_df(key='cps', to_apply=lambda x: x.sum()),
             'reaction count': result_to_df(key='rns', to_apply=lambda x: x.sum()),
             'chiral fraction': result_to_df(key='chiral_fraction', to_apply=lambda x: x)}

# plot results
fig, ax = plt.subplots(len(summaries.keys()), 1, figsize=(4, 7), sharex=True, layout='compressed')
for ict, iname in enumerate(summaries.keys()):
    summaries[iname].plot(ax=ax[ict], ylabel=iname, grid=True, legend=False, c='k', alpha=0.3,  xlabel='Expansion step')
plt.show()
plt.close()